# Landing — Budgets

Extrai `finops_source.budgets` do PostgreSQL via psycopg2 e salva como **CSV** no MinIO.

- Sem Spark
- Watermark por `created_at`

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import os, io, csv, sys

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if os.path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

import psycopg2, boto3
from utils.spark_session import get_watermark, update_watermark

# Conexão Postgres
def pg_conn():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST", "postgres"),
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        dbname=os.getenv("POSTGRES_DB", "finops"),
        user=os.getenv("POSTGRES_USER", "finops_user"),
        password=os.getenv("POSTGRES_PASSWORD", "finops_pass"),
    )

# Cliente MinIO
def s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin123"),
    )

def rows_to_csv(rows, columns):
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow({col: (val.isoformat() if hasattr(val, "isoformat") else val)
                         for col, val in zip(columns, row)})
    return buf.getvalue()

BUCKET = os.getenv("MINIO_BUCKET", "datalake")


In [ ]:
ENTITY = "budgets"
watermark = get_watermark(ENTITY)
print(f"[landing_budgets] Watermark: {watermark}")

with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT id, team_id, year, month, provider, amount_usd, created_at "
            "FROM finops_source.budgets WHERE created_at > %s",
            (watermark,),
        )
        columns = [d[0] for d in cur.description]
        rows = cur.fetchall()

if not rows:
    print(f"[landing_budgets] Nenhum registro novo. Encerrando.")
else:
    csv_data = rows_to_csv(rows, columns)
    key = f"landing/budgets/budgets_{execution_date}.csv"
    s3_client().put_object(Bucket=BUCKET, Key=key, Body=csv_data.encode())

    new_wm = max(r[-1] for r in rows)
    update_watermark(ENTITY, new_wm, row_count=len(rows))
    print(f"[landing_budgets] {len(rows)} registros → s3://{BUCKET}/{key}")
    print(f"[landing_budgets] Watermark → {new_wm}")
